In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

txn = spark.table("fintech_fraud_risk.silver.transactions")

# 1. Dedup - keep first by transaction_id
w = Window.partitionBy("transaction_id").orderBy(F.monotonically_increasing_id())
deduped = txn.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

#2. Type Conversion + amount Validity
typed = deduped.withColumn("amount", F.col("amount").cast("decimal(12,3)"))
rejected_amount = typed.filter((F.col("amount").isNull()) | (F.col("amount") <= 0)).withColumn("rejection_reason", F.lit("invalid_amount"))
valid_amount = typed.filter(F.col("amount") > 0)

#3. Timestamp standardization
valid_amount = valid_amount.withColumn(
    "transaction_timestamp", F.to_timestamp("transaction_timestamp")
)

#4. standardize cateforicals
valid_amount = (valid_amount
                .withColumn("payment_method", F.upper(F.trim("payment_method")))
                .withColumn("transaction_status", F.initcap(F.trim("transaction_status"))))

# 5. Referntial integrity - anti-join to find orphans, then split
customers = spark.table("fintech_fraud_risk.bronze.bronze_customers").select("customer_id")
merchants = spark.table("fintech_fraud_risk.bronze.bronze_merchants").select("merchant_id")
devices = spark.table("fintech_fraud_risk.bronze.bronze_devices").select("device_id")

orphan_cust = valid_amount.join(customers, "customer_id", "left_anti").withColumn("rejection_reason", F.lit("orphan_customer_id"))
orphan_merch = valid_amount.join(merchants, "merchant_id", "left_anti").withColumn("rejection_reason", F.lit("orphan_merchant_id"))
orphan_dev = valid_amount.join(devices, "device_id", "left_anti").withColumn("rejection_reason", F.lit("orphan_device_id"))

rejected_fk = orphan_cust.unionByName(orphan_merch).unionByName(orphan_dev).dropDuplicates(["transaction_id"])
silver_valid = valid_amount.join(rejected_fk.select("transaction_id"), "transaction_id", "left_anti")

# write both - never silently drop
silver_valid.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("fintech_fraud_risk.silver.transactions")
rejected_amount.unionByName(rejected_fk, allowMissingColumns=True).write.format("delta").mode("overwrite").option("overwriteSchema", "true")\
    .saveAsTable("fintech_fraud_risk.silver.transactions_rejected")

print(f"Bronze: {txn.count():,} -> silver: {silver_valid.count():,}"
      f"| Rejected: {rejected_amount.count() + rejected_fk.count():,}")

In [0]:
# Check rejected table schema and data

# rejected_df = spark.table("fintech_fraud_risk.silver.transactions_rejected")
# print("Rejected table row count:", rejected_df.count())
# print("\nSchema (note rejection_reason column):")
# rejected_df.printSchema()
# print("\nRejection reasons breakdown:")
# rejected_df.groupBy("rejection_reason").count().display()

In [0]:
#from pyspark.sql import functions as F

# customers
silver_customers = (
    spark.table("fintech_fraud_risk.bronze.bronze_customers")
    .withColumn("occupation", F.coalesce(F.trim("occupation"), F.lit("Unknown")))
    .withColumn("city", F.initcap(F.trim("city")))
    .withColumn("account_open_date", F.to_date("account_open_date"))
    .dropDuplicates(["customer_id"])
    .filter(F.col("age").between(18, 100))
)
silver_customers.write.format("delta").mode("overwrite").saveAsTable("fintech_fraud_risk.silver.customers")
